# PUMA Nucleus Segmentation - Colab Training (GPU G4 optimized)

## Tổng quan
- **GPU**: G4 (100GB VRAM) - sử dụng full capacity
- **Track**: 1 (3 classes: tumor, lymphocyte, other) hoặc 2 (10 classes)
- **Models**:
  - Segmentation: CP4 (Cellpose-SAM) với fine-tuning
  - Classification: ResNet-18 backbone
- **Augmentation**: Full Albumentations pipeline


In [ ]:
# Giải nén và xóa file zip!
!pip install -q gdown
!gdown --id 1kjyoajJtNaX0gAjPcJVmZHS8wm6IoG0M -O data.zip
!unzip -q data.zip -d extracted_data/
!rm data.zip

In [ ]:
# @title ## 1. Cấu hình Hyperparameters (EDIT HERE)
TRACK = 1  # @param [1, 2] {type:"raw", allow entries:["1","2"]}

# Training hyperparameters
SEG_EPOCHS = 100  # @param {type:"integer", min:10, max:500}
SEG_BATCH_SIZE = 8  # @param {type:"integer", min:1, max:32}
SEG_LR = 1e-5  # @param {type:"number"}
SEG_WEIGHT_DECAY = 0.1  # @param {type:"number"}
SEG_DIAMETER = 17.0  # @param {type:"number"}

CLS_PHASE1_EPOCHS = 20  # @param {type:"integer"}
CLS_PHASE2_EPOCHS = 80  # @param {type:"integer"}
CLS_BATCH_SIZE = 64  # @param {type:"integer"}
CLS_LR_HEAD = 1e-4  # @param {type:"number"}
CLS_LR_BACKBONE = 1e-6  # @param {type:"number"}

# Data split
VAL_SPLIT = 0.15  # @param {type:"number"}
TEST_SPLIT = 0.10  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}
# Paths on Google Drive
DRIVE_ROOT = "/content/drive/MyDrive/PUMA_experiments"  # @param {type:"string"}
# DATASET_ROOT = "/content/drive/MyDrive/dataset_PUMA"  # @param {type:"string"}
DATASET_ROOT = "/content/extracted_data"

print(f"Track: {TRACK}")
print(f"Seg epochs: {SEG_EPOCHS}, batch: {SEG_BATCH_SIZE}")
print(f"Cls phase1: {CLS_PHASE1_EPOCHS}, phase2: {CLS_PHASE2_EPOCHS}")
print(f"Drive root: {DRIVE_ROOT}")

## 2. Cài đặt môi trường

In [ ]:
# Clone repo
REPO_URL = "https://github.com/hoangtung386/PUMA_Nucleus_Segmentation.git"
REPO_DIR = "/content/PUMA_Nucleus_Segmentation"

!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!ls -la

In [ ]:
# Install dependencies
!pip -qqq install -U pip
!pip -qqq install -e .[dev,cellpose]
!pip -qqq install -r requirements.txt
!pip -qqq install wandb huggingface-hub
print("Dependencies installed!")

## Tạo secret key với key đặt là HF_TOKEN và paste API vào value , làm tương tự vậy với WANDB_TOKEN
### Sau đó mở quyền truy cập secret key

In [ ]:
# HuggingFace login (optional - required for private models or higher rate limits)
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("HuggingFace logged in!")
else:
    print("No HF_TOKEN, using anonymous access (may have lower rate limits)")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 3. Thiết lập đường dẫn

In [ ]:
import sys
from pathlib import Path
import shutil

# Add project to path
sys.path.insert(0, f"{REPO_DIR}/src")
sys.path.insert(0, f"{REPO_DIR}/scripts")

# Paths
REPO_ROOT = Path(REPO_DIR)
DATASET_ROOT = Path(DATASET_ROOT)
DRIVE_ROOT = Path(DRIVE_ROOT)

# Raw data paths (from dataset_PUMA)
RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
SPLIT_DIR = REPO_ROOT / "data" / "splits"

# Output paths
SAVE_DIR = DRIVE_ROOT / "models"
LOG_DIR = DRIVE_ROOT / "runs"
OUT_DIR = DRIVE_ROOT / "outputs"

# Create directories
for p in [RAW_DIR / "images", RAW_DIR / "annotations", PROCESSED_DIR, SPLIT_DIR, SAVE_DIR, LOG_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"SAVE_DIR: {SAVE_DIR}")

## 4. Chuẩn bị Dataset

In [ ]:
# Kiểm tra dataset gốc
import os

# Dataset PUMA expected structure
img_src_dir = DATASET_ROOT / "01_training_dataset_tif_ROIs"
ann_src_dir = DATASET_ROOT / "01_training_dataset_geojson_nuclei"

print(f"Image source exists: {img_src_dir.exists()}")
print(f"Annotation source exists: {ann_src_dir.exists()}")

if img_src_dir.exists():
    n_images = len(list(img_src_dir.glob("*.tif")))
    print(f"Source images: {n_images}")

In [ ]:
# Copy data to raw format
from tqdm import tqdm

img_dst_dir = RAW_DIR / "images"
ann_dst_dir = RAW_DIR / "annotations"

# Clear existing
for p in img_dst_dir.glob("*"): p.unlink()
for p in ann_dst_dir.glob("*"): p.unlink()

copied, missing = 0, []
for img_path in tqdm(sorted(img_src_dir.glob("*.tif")), desc="Copying"):
    stem = img_path.stem
    ann_path = ann_src_dir / f"{stem}_nuclei.geojson"
    if not ann_path.exists():
        missing.append(stem)
        continue
    shutil.copy2(img_path, img_dst_dir / f"{stem}.tif")
    shutil.copy2(ann_path, ann_dst_dir / f"{stem}.geojson")
    copied += 1

print(f"Copied: {copied} pairs")
print(f"Missing: {len(missing)}")
print(f"Images: {len(list(img_dst_dir.glob('*.tif')))}")
print(f"Annotations: {len(list(ann_dst_dir.glob('*.geojson')))}")

In [ ]:
# Process data (parse GeoJSON -> masks)
import yaml
import json
import numpy as np

from puma_seg.data.geojson_parser import parse_geojson, get_class_map, NUM_CLASSES
from puma_seg.utils.io_utils import list_image_paths
from scripts.prepare_data import make_split, process_one

# Verify track
n_fg = NUM_CLASSES[TRACK] - 1
print(f"Track {TRACK}: {n_fg} foreground classes")

# Get image paths
img_dir = RAW_DIR / "images"
ann_dir = RAW_DIR / "annotations"
image_paths = list_image_paths(img_dir, extensions=(".tif",))
stems = [p.stem for p in image_paths]
print(f"Found {len(stems)} images")

# Make split
split = make_split(stems, VAL_SPLIT, TEST_SPLIT, SEED)
print(f"Split: {len(split['train'])} train / {len(split['val'])} val / {len(split['test'])} test")

In [ ]:
# Save split and process each image
# Save split file
split_path = SPLIT_DIR / "split.json"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
with open(split_path, "w", encoding="utf-8") as f:
    json.dump(split, f, indent=2)

# Process each subset
stem_to_path = {p.stem: p for p in image_paths}
for subset in ["train", "val", "test"]:
    out_subset = PROCESSED_DIR / subset
    print(f"Processing {subset}...")
    for stem in tqdm(split[subset], desc=subset):
        img_path = stem_to_path.get(stem)
        if img_path:
            process_one(img_path, ann_dir, out_subset, TRACK, ".tif")

# Verify
for subset in ["train", "val", "test"]:
    n_imgs = len(list((PROCESSED_DIR / subset / "images").glob("*")))
    n_masks = len(list((PROCESSED_DIR / subset / "masks").glob("*")))
    n_labels = len(list((PROCESSED_DIR / subset / "labels").glob("*")))
    print(f"{subset}: {n_imgs} images, {n_masks} masks, {n_labels} labels")

## 5. Tạo Config

In [ ]:
import yaml
from datetime import datetime

run_name = f"track{TRACK}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"Run name: {run_name}")

# Create config
cfg = {
    "data": {
        "raw_dir": str(RAW_DIR),
        "processed_dir": str(PROCESSED_DIR),
        "split_file": str(SPLIT_DIR / "split.json"),
        "track": TRACK,
        "image_size": None,
        "val_split": VAL_SPLIT,
        "test_split": TEST_SPLIT,
        "crop_size": 64,
    },
    "segmentation": {
        "pretrained_model": "cpsam",
        "gpu": True,
        "diameter": SEG_DIAMETER,
        "nchan": 2,
        "channels": [0, 0],
        "learning_rate": SEG_LR,
        "weight_decay": SEG_WEIGHT_DECAY,
        "n_epochs": SEG_EPOCHS,
        "batch_size": SEG_BATCH_SIZE,
        "model_name": f"puma_cellpose_{run_name}",
    },
    "classification": {
        "pretrained": True,
        "freeze_backbone": True,
        "dropout": 0.3,
        "phase1_epochs": CLS_PHASE1_EPOCHS,
        "phase1_lr": 1e-3,
        "phase2_epochs": CLS_PHASE2_EPOCHS,
        "phase2_lr_head": CLS_LR_HEAD,
        "phase2_lr_backbone": CLS_LR_BACKBONE,
        "unfreeze_groups": 2,
        "weight_decay": 1e-4,
        "label_smoothing": 0.1,
        "patience": 15,
        "batch_size": CLS_BATCH_SIZE,
        "use_amp": True,
        "model_name": f"best_classifier_{run_name}",
    },
    "paths": {
        "save_dir": str(SAVE_DIR / run_name),
        "log_dir": str(LOG_DIR / run_name),
        "output_dir": str(OUT_DIR / run_name),
    },
}

# Save config
cfg_path = REPO_ROOT / "configs" / f"{run_name}.yaml"
cfg_path.parent.mkdir(parents=True, exist_ok=True)
with open(cfg_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Config saved: {cfg_path}")

## 6. Training

In [ ]:
# W&B init (optional)
try:
    import wandb
    from google.colab import userdata
    WANDB_TOKEN = userdata.get('WANDB_TOKEN')
    if WANDB_TOKEN:
        wandb.login(key=WANDB_TOKEN)
        wandb_run = wandb.init(
            project='puma-nucleus-segmentation',
            name=run_name,
            config=cfg,
            job_type='train',
        )
        print("W&B initialized!")
    else:
        print("No WANDB_TOKEN, skipping...")
        wandb_run = None
except Exception as e:
    print(f"W&B error: {e}")
    wandb_run = None

In [ ]:
# Segmentation Training
import argparse
import torch

from puma_seg.cli._train_impl import load_config, run_segmentation

print("=" * 50)
print("TRAINING SEGMENTATION (CP4)")
print("=" * 50)

# Check GPU
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load config
cfg = load_config(cfg_path)

# Training args
args = argparse.Namespace(
    config=cfg_path,
    mode='segmentation',
    resume_seg=None,
    resume_cls=None,
    dry_run=False,
)

# Run
seg_model_path = run_segmentation(cfg, args)
print(f"Segmentation model saved: {seg_model_path}")

In [ ]:
# Classification Training
from puma_seg.cli._train_impl import load_config, run_classification

print("=" * 50)
print("TRAINING CLASSIFICATION (ResNet-18)")
print("=" * 50)

# Load config
cfg = load_config(cfg_path)

# Training args
args = argparse.Namespace(
    config=cfg_path,
    mode='classification',
    resume_seg=None,
    resume_cls=None,
    dry_run=False,
)

# Run
run_classification(cfg, args)
print("Classification training done!")

## 7. Evaluation

In [ ]:
# Load models for evaluation
import torch
import cv2
import numpy as np

from pathlib import Path
from puma_seg.models.cellpose_wrapper import CellposeSegmentor
from puma_seg.models.nucleus_classifier import NucleusClassifier
from puma_seg.data.geojson_parser import (
    get_class_names, get_nucleus_centroids, extract_nucleus_crops
)
from puma_seg.data.transforms import get_crop_val_transforms
from puma_seg.evaluation.metrics import evaluate_predictions, compute_puma_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Load models
from pathlib import Path

# Segmentation model
seg_model_path = SAVE_DIR / run_name / f"puma_cellpose_{run_name}.pth"
if seg_model_path.exists():
    segmentor = CellposeSegmentor(
        pretrained_model=str(seg_model_path),
        gpu=True,
        diameter=SEG_DIAMETER,
    )
    print(f"Loaded segmentor from {seg_model_path}")
else:
    segmentor = CellposeSegmentor(
        pretrained_model="cpsam",
        gpu=True,
        diameter=SEG_DIAMETER,
    )
    print("Using pretrained cpsam")

# Classification model
cls_model_path = SAVE_DIR / run_name / f"best_classifier_{run_name}.pth"
if cls_model_path.exists():
    classifier = NucleusClassifier.load(cls_model_path)
    classifier.to(device).eval()
    print(f"Loaded classifier from {cls_model_path}")
    cls_transform = get_crop_val_transforms(64)
else:
    print("No classifier checkpoint found")
    classifier = None
    cls_transform = None

In [ ]:
# Run evaluation on test set
from puma_seg.data.dataset import PUMASegmentationDataset
from puma_seg.utils.io_utils import load_mask, list_image_paths
from tqdm import tqdm

# Load test dataset
test_ds = PUMASegmentationDataset(
    processed_dir=PROCESSED_DIR,
    split="test",
    track=TRACK,
)

class_names = get_class_names(TRACK)
n_fg = len(class_names) - 1
print(f"Test set: {len(test_ds)} images")
print(f"Classes: {class_names[1:]}")

# Evaluate each image
pred_list, gt_list = [], []

for idx in tqdm(range(len(test_ds)), desc="Evaluating"):
    image, gt_mask = test_ds[idx]
    img_path = test_ds.image_paths[idx]
    img_stem = img_path.stem
    
    # GT labels
    gt_label_path = PROCESSED_DIR / "test" / "labels" / f"{img_stem}.npy"
    if gt_label_path.exists():
        gt_instance_classes = np.load(gt_label_path)
        gt_instance_labels = {i + 1: int(gt_instance_classes[i]) for i in range(len(gt_instance_classes))}
    else:
        gt_instance_labels = {}
    
    gt_centroids = get_nucleus_centroids(gt_mask)
    
    # Predict
    pred_mask, info = segmentor.predict(image, diameter=SEG_DIAMETER)
    pred_centroids = info["centroids"]
    
    # Classify
    pred_classes = {}
    if classifier is not None and pred_mask.max() > 0:
        crops, inst_ids = extract_nucleus_crops(image, pred_mask, crop_size=64)
        if crops:
            tensors = torch.stack([cls_transform(image=crop)["image"] for crop in crops]).to(device)
            with torch.no_grad():
                logits = classifier(tensors)
                class_preds = logits.argmax(dim=1).cpu().tolist()
            pred_classes = {inst_id: cls_pred + 1 for inst_id, cls_pred in zip(inst_ids, class_preds)}
    else:
        pred_classes = {instance_id: 1 for instance_id in pred_centroids}
    
    pred_list.append({"centroids": pred_centroids, "classes": pred_classes})
    gt_list.append({"centroids": gt_centroids, "classes": gt_instance_labels})

print(f"\nEvaluated {len(pred_list)} images")

In [ ]:
# Compute metrics
metrics = evaluate_predictions(
    pred_list,
    gt_list,
    n_classes=n_fg,
    class_names=class_names[1:],
    threshold=15.0,
)

print("=" * 60)
print("PUMA EVALUATION RESULTS")
print("=" * 60)
for key, value in metrics.items():
    print(f"  {key:25s}: {value:.4f}")
print("=" * 60)

if wandb_run:
    wandb.log(metrics)
print("Metrics computed!")

## 8. Visualization

In [ ]:
# Sample predictions
import matplotlib.pyplot as plt
from puma_seg.utils.visualization import plot_sample, plot_predictions_vs_gt

# Load sample
sample_idx = 0
image, instance_mask = test_ds[sample_idx]
img_stem = test_ds.image_paths[sample_idx].stem

# Predict
pred_mask, info = segmentor.predict(image, diameter=SEG_DIAMETER)
print(f"GT nuclei: {instance_mask.max()}")
print(f"Pred nuclei: {pred_mask.max()}")

# Load GT labels
gt_label_path = PROCESSED_DIR / "test" / "labels" / f"{img_stem}.npy"
class_mask = None
if gt_label_path.exists():
    class_mask = np.load(gt_label_path)

# Plot
plot_sample(image, instance_mask, class_mask, track=TRACK, title=f"Test sample {sample_idx} - GT")
plt.show()

plot_predictions_vs_gt(image, pred_mask, instance_mask, None, track=TRACK, title=f"Test sample {sample_idx} - Prediction")
plt.show()

## 9. Save Artifacts

In [ ]:
# Save checkpoints to Drive
from pathlib import Path

save_dir = Path(cfg["paths"]["save_dir"])
log_dir = Path(cfg["paths"]["log_dir"])
output_dir = Path(cfg["paths"]["output_dir"])

print(f"save_dir: {save_dir}")
print(f"log_dir: {log_dir}")
print(f"output_dir: {output_dir}")

# List files
print("\nSaved files:")
for p in save_dir.glob("*"):
    print(f"  {p.name}")
    
if wandb_run:
    artifact = wandb.Artifact(name=f"puma-checkpoints-{run_name}", type="model")
    if save_dir.exists():
        artifact.add_dir(str(save_dir))
    wandb.log_artifact(artifact)
    wandb.finish()
print("\nArtifacts saved!")

In [ ]:
print("Sau khi hoàn thành") 
print("Model checkpoint sẽ được lưu vào:")
print(f"{SAVE_DIR} / {run_name}")
print("Các tệp W&B (nếu em bật)")